# Utilitaires tenseurs

> Conversions et extraction de tenseurs depuis les sorties de modèles.


In [ ]:
#| default_exp tensors


In [ ]:
#| export
"""Tensor conversion helpers used by adapters, collectors, and plots."""


from typing import Any

import numpy as np
import torch
from torch import Tensor


def to_numpy(value: Any) -> np.ndarray:
    """Convert tensors and array-like values to a NumPy array."""

    if isinstance(value, Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def select_tensor(value: Any, index: int = 0) -> Tensor:
    """Extract a tensor from common model return shapes."""

    if isinstance(value, Tensor):
        return value
    if isinstance(value, dict):
        for key in ("z", "latent", "latents", "embedding", "features"):
            if key in value:
                return select_tensor(value[key], index=index)
        raise KeyError(
            "Encoder returned a dict without one of: z, latent, latents, embedding, features."
        )
    if isinstance(value, (tuple, list)):
        return select_tensor(value[index], index=index)
    raise TypeError(f"Cannot extract a Tensor from {type(value)!r}.")


def move_to_device(value: Any, device: torch.device | str) -> Any:
    """Move nested tensor containers to a device."""

    if isinstance(value, Tensor):
        return value.to(device)
    if isinstance(value, dict):
        return {key: move_to_device(item, device) for key, item in value.items()}
    if isinstance(value, tuple):
        return tuple(move_to_device(item, device) for item in value)
    if isinstance(value, list):
        return [move_to_device(item, device) for item in value]
    return value


def detach_cpu(value: Any) -> Tensor:
    """Extract a tensor if needed, detach it, and move it to CPU."""

    tensor = select_tensor(value) if not isinstance(value, Tensor) else value
    return tensor.detach().cpu()
